In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.cluster import KMeans

In [ ]:
df = pd.read_csv("Gold_Price_financial.csv")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop(columns=["Date"])
df = df.fillna(df.mean())
df.isnull().sum()

In [ ]:
print(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.shape

In [ ]:
X = df.drop(columns=["GLD"])
y = df["GLD"]

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

In [ ]:
selector = SelectKBest(score_func=f_regression, k=3)
selector.fit(X_scaled, y)
print(pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False))
selected = X.columns[selector.get_support()]
selected

In [ ]:
X_sel = X_scaled[selected]
X_train, X_test, y_train, y_test = train_test_split(X_sel, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [ ]:
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return [mae, mse, rmse, r2]

In [ ]:
results = pd.DataFrame({
    "Decision Tree": evaluate(y_test, dt_pred),
    "Random Forest": evaluate(y_test, rf_pred)
}, index=["MAE", "MSE", "RMSE", "R2"])
results

In [ ]:
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)
plt.plot(range(1, 11), inertia, marker="o")
plt.show()

In [ ]:
sil = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil.append(silhouette_score(X_scaled, labels))
plt.plot(range(2, 11), sil, marker="o")
plt.show()

In [ ]:
best_k = range(2, 11)[np.argmax(sil)]
best_k

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)
df["Cluster"].value_counts()

In [ ]:
df.groupby("Cluster")["GLD"].describe()

In [ ]:
df.boxplot(column="GLD", by="Cluster")
plt.show()

In [ ]:
plt.scatter(X_scaled.iloc[:, 0], X_scaled.iloc[:, 1], c=df["Cluster"], cmap="viridis")
plt.show()

In [ ]:
dt_params = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}
dt_grid = GridSearchCV(DecisionTreeRegressor(random_state=42), dt_params, cv=5, scoring="r2")
dt_grid.fit(X_train, y_train)
print(dt_grid.best_params_)

In [ ]:
rf_params = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5]
}
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42), rf_params, cv=5, scoring="r2")
rf_grid.fit(X_train, y_train)
print(rf_grid.best_params_)

In [ ]:
dt_best_pred = dt_grid.best_estimator_.predict(X_test)
rf_best_pred = rf_grid.best_estimator_.predict(X_test)
final = pd.DataFrame({
    "Decision Tree": evaluate(y_test, dt_pred),
    "Random Forest": evaluate(y_test, rf_pred),
    "Tuned Decision Tree": evaluate(y_test, dt_best_pred),
    "Tuned Random Forest": evaluate(y_test, rf_best_pred)
}, index=["MAE", "MSE", "RMSE", "R2"])
final

In [ ]:
plt.scatter(y_test, dt_best_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
plt.show()

In [ ]:
plt.scatter(y_test, rf_best_pred)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red")
plt.show()

In [ ]:
plt.scatter(dt_best_pred, y_test - dt_best_pred)
plt.axhline(0, color="red")
plt.show()

In [ ]:
plt.scatter(rf_best_pred, y_test - rf_best_pred)
plt.axhline(0, color="red")
plt.show()

In [ ]:
final.T[["MAE", "RMSE"]].plot(kind="bar")
plt.show()

In [ ]:
final.T["R2"].plot(kind="bar")
plt.show()